# Sweep de Decomposição — Tucker vs Tensor-Train (ResNet-18 + ablation ensemble, SEM fine-tuning)

Este notebook une os dois pipelines de compressão estrutural que antes viviam separados (`Tucker/` e `TensorTrain/`). Os dois métodos usam a mesma heurística de rank por energia do SVD, o mesmo baseline, split de teste e métricas — a única coisa que muda entre eles é a função de decomposição aplicada a cada camada convolucional elegível (conv 3x3, sem grupos, `in_channels > 64`). Quantização (fp16/int8) foi deixada de fora propositalmente.

**Sem fine-tuning:** depois de decompor o modelo em cada threshold, ele é avaliado diretamente no conjunto de teste, com os pesos reconstruídos pela decomposição (zero-shot) — não há nenhuma etapa de re-treino/ajuste fino. `test_f1`/`test_auc` no CSV refletem, portanto, o efeito puro da decomposição sobre a acurácia, sem nenhuma recuperação via treino.

**Mesmo sistema de seed que os notebooks com Optuna:** o modelo base é a **ResNet-18** treinada por `pipeline_resnet18.ipynb` — a ablation de scheduler com hiperparâmetros **fixos** (não Optuna) e avaliação por **ensemble dos 5 folds** no teste. A seção 1b (`select_best_seed_per_scheduler`) lê `ablation_logs_resnet18.csv` e elege, por scheduler, a seed de melhor `Test_Ensemble_AUC` dentre as 7 em `SEEDS = [42, 17, 7, 12, 123, 555, 996]` — só essa seed é decomposta por scheduler (3 no total, não 21). Isso substitui o `summarize_across_seeds` da versão anterior deste notebook (que rodava as 7 seeds de uma vez e agregava média±desvio entre elas): agora a variância que importa comparar é **entre os 3 schedulers** (cada um já com seu melhor candidato), por isso a função equivalente aqui é `summarize_across_schedulers`.

In [1]:
import os
import copy
import csv
import random
import time
import sys
from datetime import datetime
from pathlib import Path

import torch
import numpy                      as     np
import torch.nn                   as     nn
import torch.nn.functional        as     F
import tensorly                   as     tl
from   torch.amp                  import autocast
from   torch.utils.data           import DataLoader, Subset
from   torchvision                import datasets, models, transforms
from   sklearn.model_selection    import train_test_split
from   sklearn.metrics            import confusion_matrix, f1_score, roc_auc_score

NOTEBOOK_DIR    = Path.cwd()
PROJECT_ROOT    = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "src").is_dir()), NOTEBOOK_DIR)  # raiz do repositório
TUCKER_DIR      = PROJECT_ROOT / "src"
TENSORTRAIN_DIR = PROJECT_ROOT / "src"
for _p in (TUCKER_DIR, TENSORTRAIN_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Cada decomposição continua no seu próprio arquivo (nada foi duplicado);
# aqui só importamos as duas funções de substituição de camada.
from tucker_compressor       import replace_layers_with_tucker
from tensor_train_decomposer import replace_layers_with_tt_tensorly, get_model_size_mb

tl.set_backend('pytorch')

# Dicionário que trata as duas decomposições como a MESMA coisa do ponto de
# vista do sweep: uma função (model, threshold_de_energia) -> model comprimido.
DECOMPOSERS = {
    "tucker":       replace_layers_with_tucker,
    "tensor_train": replace_layers_with_tt_tensorly,
}

## 1. Configuração

`METHODS` define quais decomposições rodam nesse notebook e em que ordem — hoje as duas, `tucker` e `tensor_train`. `THRESHOLDS` é a mesma lista de limiares de energia (SVD) aplicada a ambos os métodos, para que a comparação entre eles seja direta (mesmo threshold ⇒ mesma "meta" de energia retida, ranks diferentes por método).

`SCHEDULERS` e `SEEDS` definem o universo de checkpoints já treinados por `pipeline_resnet18.ipynb` (3 schedulers × 7 seeds = 21 combinações, cada uma com um checkpoint `{scheduler}_seed{seed}_fold{TARGET_FOLD}_best.pth` em `CHECKPOINT_DIR`). Este notebook não decompõe as 21 — a seção 1b (`select_best_seed_per_scheduler`) escolhe, por scheduler, a seed de melhor desempenho dentre as 7, e só essa é decomposta.

In [2]:
BATCH_SIZE   = 8
TARGET_FOLD  = 1
SCHEDULERS   = ["CosineAnnealingLR", "OneCycleLR", "ReduceLROnPlateau"]  # os 3 schedulers da ablation (ResNet-18, ensemble) — nomes exatos de pipeline_resnet18.ipynb
SEEDS        = [42, 17, 7, 12, 123, 555, 996]
SCHEDULER    = SCHEDULERS[0]                     # "corrente"; main(scheduler, seed) reatribui este global a cada iteração
SEED         = SEEDS[0]                          # idem
BOOTSTRAP_SEED = 42   # RNG do bootstrap (IC da AUC): fixo, independe da seed de dados/checkpoint em teste

BASE_DIR       = PROJECT_ROOT / "Originais"
CHECKPOINT_DIR = PROJECT_ROOT / "ablation_schedulers_resnet18_ckpt_clinical_composite"
OUTPUT_DIR     = PROJECT_ROOT
TEST_SIZE      = 0.2

METHODS    = ["tucker", "tensor_train"]     # roda os dois, nessa ordem, no mesmo notebook
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99, 0.999]

LATENCY_SHAPE          = (1, 3, 224, 224)   # "1 imagem" — usado no benchmark de latência e nos GFLOPs
SINGLE_IMAGE_THRESHOLD = 0.90               # threshold usado na comparação final: original vs Tucker vs TT

NUM_WORKERS         = 4         # No Windows, se der problema com workers, baixe para 0.
DETERMINISTIC       = False     # False = cudnn.benchmark ON (mais rápido). True = reprodutível bit-a-bit.
BOOTSTRAP_N         = 1000      # Reamostragens para o intervalo de confiança.

PRETRAINED = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()   # AMP só faz sentido (e só engata) em CUDA.

## 1b. Seleção da melhor seed por scheduler

`pipeline_resnet18.ipynb` já treinou e logou as 21 combinações `(scheduler, seed)` em `ablation_logs_resnet18.csv` (uma linha por combinação, com `Test_Ensemble_AUC` — a AUC no teste blindado do ensemble dos 5 folds daquele treino). `select_best_seed_per_scheduler` lê esse CSV e escolhe, para cada scheduler, a seed com maior `BEST_SEED_METRIC` (`Test_Ensemble_AUC` por padrão) — só essa seed é decomposta em `main()`. Isso reduz o sweep de 21 combinações `(scheduler, seed)` para **3** (uma por scheduler).

In [3]:
# --- Seleção da melhor seed por scheduler (reduz o custo do sweep de decomposição) ---
# pipeline_resnet18.ipynb já treinou e logou as 21 combinações (scheduler, seed) em
# ablation_logs_resnet18.csv — uma linha por combinação, já com a métrica de teste do
# ensemble dos 5 folds. Em vez de decompor as 7 seeds de cada scheduler, pegamos só a
# seed de MELHOR desempenho por scheduler e decompomos apenas esse modelo — 3
# decomposições no total (uma por scheduler).
ABLATION_LOG_CSV = CHECKPOINT_DIR / "ablation_logs_resnet18.csv"
BEST_SEED_METRIC  = "Test_Ensemble_AUC"   # métrica do ablation log usada para eleger a seed vencedora


def select_best_seed_per_scheduler(csv_path=ABLATION_LOG_CSV, schedulers=SCHEDULERS, seeds=SEEDS,
                                   metric=BEST_SEED_METRIC):
    """Lê o log de ablation (uma linha por (scheduler, seed) já treinado por
    pipeline_resnet18.ipynb) e retorna, para cada scheduler, a seed com maior
    `metric` dentre as seeds em SEEDS — só essa seed é decomposta em main()."""
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"{csv_path} não encontrado. Rode pipeline_resnet18.ipynb primeiro.")
    best = {}   # scheduler -> (seed, valor da métrica)
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            sch, sd = row["Scheduler"], int(row["Seed"])
            if sch not in schedulers or sd not in seeds:
                continue
            val = float(row[metric])
            if sch not in best or val > best[sch][1]:
                best[sch] = (sd, val)
    missing = [s for s in schedulers if s not in best]
    if missing:
        raise ValueError(f"Sem linha em {csv_path} para os schedulers {missing} (dentre as seeds em SEEDS).")
    print(f"[seleção] melhor seed por scheduler (métrica={metric}, dentre {seeds}):")
    for sch in schedulers:
        sd, val = best[sch]
        print(f"   {sch:<20} -> seed {sd}  ({metric}={val:.4f})")
    return {sch: sd for sch, (sd, _) in best.items()}


BEST_SEED_PER_SCHEDULER = select_best_seed_per_scheduler()

[seleção] melhor seed por scheduler (métrica=Test_Ensemble_AUC, dentre [42, 17, 7, 12, 123, 555, 996]):
   CosineAnnealingLR    -> seed 123  (Test_Ensemble_AUC=0.9789)
   OneCycleLR           -> seed 123  (Test_Ensemble_AUC=0.9833)
   ReduceLROnPlateau    -> seed 555  (Test_Ensemble_AUC=0.9842)


## 2. Reprodutibilidade & utilitários

`set_seed` centraliza a reprodutibilidade (PyTorch, NumPy, `random`, cuDNN) — agora recebe a seed explicitamente (chamada uma vez por seed do sweep, não uma constante global fixa). `count_parameters` conta parâmetros treináveis. `free_gpu` libera referências grandes e limpa o cache da GPU entre etapas do sweep. `timed_call` mede o tempo de parede de qualquer chamada, sincronizando a GPU antes/depois para não distorcer a medição com operações CUDA assíncronas.

In [4]:
def set_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Trade-off explícito: determinismo (lento) vs autotuner do cuDNN (rápido).
        torch.backends.cudnn.deterministic = DETERMINISTIC
        torch.backends.cudnn.benchmark = not DETERMINISTIC

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def free_gpu(*objs):
    for o in objs:
        del o
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def timed_call(fn, *args, **kwargs):
    """Executa fn(*args, **kwargs) e mede o tempo de parede (wall-clock).
    Sincroniza a GPU antes/depois para evitar que operações CUDA assíncronas
    ainda em voo sejam contadas fora da janela medida (ou vice-versa)."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    result = fn(*args, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start
    return result, elapsed

## 3. Modelo / transforms

`build_model` cria a ResNet-18 usada no experimento (pesos pré-treinados quando `PRETRAINED=True`, com a `fc` substituída para `num_classes` saídas) — mesma arquitetura e mesmo `build_model` de `pipeline_resnet18.ipynb`. Como não há fine-tuning, só o transform de avaliação (`teste`, sem augmentation) é usado — é o mesmo aplicado ao treino original dos checkpoints na hora de validar/testar.

In [5]:
def build_model(num_classes):
    weights  = models.ResNet18_Weights.DEFAULT if PRETRAINED else None
    model    = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

data_transforms = {
    'teste': transforms.Compose([
        transforms.Resize(size=[224, 224]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

## 4. Avaliação (multiclasse: F1 macro, AUC One-vs-Rest macro)

### Função `evaluate`
Avalia um modelo em um `DataLoader` sem calcular gradientes. Retorna F1 macro, AUC macro One-vs-Rest, matriz de confusão e os vetores brutos de rótulos, predições e probabilidades. Usada tanto no baseline quanto em cada ponto do sweep de qualquer um dos dois métodos.

In [6]:
@torch.no_grad()
def evaluate(model, loader, num_classes):
    model.eval()
    labels_all, preds_all, probs_all = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)

        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)

        preds = torch.argmax(outputs, dim=1)
        probs = F.softmax(outputs.float(), dim=1)

        labels_all.extend(labels.numpy())
        preds_all.extend(preds.cpu().numpy())
        probs_all.extend(probs.cpu().numpy())

    y = np.array(labels_all)
    p = np.array(preds_all)
    s = np.array(probs_all)

    f1 = f1_score(y, p, average='macro')
    try:
        auc = roc_auc_score(y, s, multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')

    cm = confusion_matrix(y, p, labels=range(num_classes))
    return dict(f1=f1, auc=auc, cm=cm, y=y, preds=p, probs=s)

## 5. Bootstrap — IC da AUC e ΔAUC pareado

`bootstrap_auc_ci` estima o intervalo de confiança da AUC por bootstrap. `paired_delta_auc_ci` calcula o IC da diferença pareada de AUC entre o modelo comprimido (Tucker ou TT) e o baseline, reamostrando os MESMOS índices para os dois — se o IC não cruzar 0, a diferença é estatisticamente significativa.

In [7]:
def bootstrap_auc_ci(y, s, num_classes, n=BOOTSTRAP_N, seed=BOOTSTRAP_SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); vals = []
    for _ in range(n):
        bi = rng.choice(idx, size=len(idx), replace=True)
        yb = y[bi]
        if len(np.unique(yb)) < num_classes:
            continue
        vals.append(roc_auc_score(yb, s[bi], multi_class='ovr', average='macro'))
    if not vals:
        return (float('nan'), float('nan'))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


def paired_delta_auc_ci(y, base_s, comp_s, num_classes, n=BOOTSTRAP_N, seed=BOOTSTRAP_SEED):
    """IC do ΔAUC = AUC(comprimido) - AUC(baseline) reamostrando os MESMOS índices."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); deltas = []
    for _ in range(n):
        bi = rng.choice(idx, size=len(idx), replace=True)
        yb = y[bi]
        if len(np.unique(yb)) < num_classes:
            continue
        auc_comp = roc_auc_score(yb, comp_s[bi], multi_class='ovr', average='macro')
        auc_base = roc_auc_score(yb, base_s[bi], multi_class='ovr', average='macro')
        deltas.append(auc_comp - auc_base)
    if not deltas:
        return (float('nan'), float('nan'), False)
    lo, hi = np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)
    sig = (lo > 0) or (hi < 0)
    return (float(lo), float(hi), bool(sig))

## 6. Erro de reconstrução por camada (genérico: Tucker ou Tensor-Train)

### Função `per_layer_reconstruction_error`
Calcula o erro relativo de reconstrução (Frobenius, %) de cada camada convolucional substituída, comparando o peso original com o peso reconstruído a partir da decomposição. É a mesma métrica dos dois pipelines separados, só que agora recebe `method` para saber como reconstruir o kernel denso: a partir dos núcleos TT (`core_1..core_4` + `tl.tt_to_tensor`) ou a partir dos três fatores Tucker (`reduce`/`core`/`expand` + `tl.tucker_to_tensor`).

In [8]:
def per_layer_reconstruction_error(original_model, compressed_model, method):
    # Snapshot dos pesos originais das camadas elegíveis (mesmo filtro dos dois métodos)
    snap = {n: m.weight.detach().cpu().clone()
            for n, m in original_model.named_modules()
            if isinstance(m, nn.Conv2d)
                            and m.kernel_size == (3, 3)
                            and m.groups == 1
                            and m.in_channels > 64}
    errs = {}
    for name, mod in compressed_model.named_modules():
        if name not in snap:
            continue

        if method == "tensor_train" and hasattr(mod, 'core_1') and hasattr(mod, 'core_4'):
            W = tl.tt_to_tensor([mod.core_1, mod.core_2, mod.core_3, mod.core_4]).detach().cpu()
        elif method == "tucker" and hasattr(mod, 'reduce') and hasattr(mod, 'expand'):
            C_out, r_out = mod.expand.weight.shape[0], mod.expand.weight.shape[1]
            r_in, C_in   = mod.reduce.weight.shape[0], mod.reduce.weight.shape[1]
            U_out = mod.expand.weight.detach().view(C_out, r_out).cpu()
            U_in  = mod.reduce.weight.detach().view(r_in, C_in).t().contiguous().cpu()
            core  = mod.core.weight.detach().cpu()
            W = tl.tucker_to_tensor((core, [U_out, U_in]))
        else:
            continue

        w0 = snap[name]
        errs[name] = (torch.linalg.norm(w0 - W) / torch.linalg.norm(w0) * 100).item()
    return errs

## 7. Latência (benchmark correto) e GFLOPs

O sweep anterior media "tempo de teste" como o wall-clock do `evaluate()` sobre o test loader inteiro. Isso mistura I/O de disco, overhead do DataLoader e principalmente o autotuning do cuDNN (`cudnn.benchmark=True`), que só é pago na PRIMEIRA vez que um shape aparece — como o baseline roda primeiro no notebook, ele sozinho paga esse custo, e todo o sweep seguinte parece "instantâneo" por comparação (não é ganho real de FLOPs). As duas funções abaixo substituem essa medição por uma correta.

### Função `benchmark_latency`
A mesma dos scripts do Tucker (`Tucker/tucker_compressor.py` / `files/tucker_compressor.py`): faz alguns forwards de warmup (não cronometrados, para o cuDNN escolher e cachear o algoritmo), sincroniza a GPU antes de cronometrar, e mede cada passada com `torch.cuda.Event` (ou `time.perf_counter` em CPU). Retorna mediana/média/desvio em ms — usada aqui sempre com `input_shape=(1,3,224,224)`, ou seja, latência de **1 imagem**.

### Funções `count_macs` / `macs_to_gflops`
Contam MACs via hooks de forward em cada `nn.Conv2d` (cobre o baseline e os 3 sub-convs REAIS do Tucker: `reduce`/`core`/`expand`). O Tensor-Train é tratado à parte: `TTConv2dTensorly` reconstrói o kernel denso `[C_out, C_in, Kh, Kw]` a partir dos núcleos e só então convolui — não existe um `nn.Conv2d` real ali dentro. Por isso o hook do TT calcula o custo como se fosse a conv densa original: o TT economiza parâmetros/disco, mas **não** economiza FLOPs de inferência (ao contrário do Tucker, que de fato reduz compute). `GFLOPs = MACs × 2 / 1e9`.

In [9]:
@torch.no_grad()
def benchmark_latency(model, dev, input_shape=(1, 3, 224, 224), warmup=10, runs=50, use_amp=False):
    """Latência de inferência medida CORRETAMENTE: warmup + sincronização CUDA.
    Sem warmup, a 1ª passada inclui autotuning do cuDNN e inicialização lazy,
    o que infla artificialmente qualquer modelo medido primeiro. Retorna ms."""
    model.eval()
    x = torch.randn(*input_shape, device=dev)
    use_cuda = (dev.type == 'cuda')

    for _ in range(warmup):
        with autocast('cuda', enabled=use_amp and use_cuda):
            model(x)
    if use_cuda:
        torch.cuda.synchronize()

    times = []
    for _ in range(runs):
        if use_cuda:
            s = torch.cuda.Event(enable_timing=True)
            e = torch.cuda.Event(enable_timing=True)
            s.record()
            with autocast('cuda', enabled=use_amp):
                model(x)
            e.record()
            torch.cuda.synchronize()
            times.append(s.elapsed_time(e))   # ms
        else:
            t0 = time.perf_counter()
            model(x)
            times.append((time.perf_counter() - t0) * 1000)

    t = np.array(times)
    return dict(median_ms=float(np.median(t)), mean_ms=float(t.mean()), std_ms=float(t.std()))


def count_macs(model, dev, input_shape=(1, 3, 224, 224)):
    """Soma MACs via hooks de forward. nn.Conv2d comuns (baseline + reduce/core/expand
    do Tucker) usam o shape real da conv. TTConv2dTensorly não tem nn.Conv2d interno
    (reconstrói o kernel denso e chama F.conv2d direto), então é tratado à parte com
    o mesmo custo de uma conv densa equivalente — sem redução de FLOPs."""
    macs = [0]
    handles = []

    def conv_hook(mod, inp, out):
        Co = mod.out_channels
        Ci = mod.in_channels // mod.groups
        Kh, Kw = mod.kernel_size
        macs[0] += Co * Ci * Kh * Kw * out.shape[2] * out.shape[3]

    def tt_hook(mod, inp, out):
        Co, Ci = mod.core_1.shape[1], mod.core_2.shape[1]
        Kh, Kw = mod.core_3.shape[1], mod.core_4.shape[1]
        macs[0] += Co * Ci * Kh * Kw * out.shape[2] * out.shape[3]

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            handles.append(m.register_forward_hook(conv_hook))
        elif hasattr(m, 'core_1') and hasattr(m, 'core_4'):
            handles.append(m.register_forward_hook(tt_hook))

    model.eval()
    with torch.no_grad():
        model(torch.randn(*input_shape, device=dev))
    for h in handles:
        h.remove()
    return macs[0]


def macs_to_gflops(macs):
    return macs * 2 / 1e9

## 8. Sweep genérico de thresholds (roda para qualquer método) — com resume, SEM fine-tuning

### Funções `log_row` / `get_completed_thresholds` / `load_completed_rows`
Persistência incremental do sweep, no mesmo padrão de resume usado nos notebooks `pipeline_resnetXX*.ipynb`: cada linha (um threshold de um método, para um `(scheduler, seed)`) é gravada em `decomposition_sweep_pareto_resnet18_ensemble_no_ft.csv` assim que fica pronta, e `run_sweep` pula qualquer `(method, threshold, seed, scheduler)` que já esteja no CSV.

### Função `run_sweep`
Parametrizada por `method` (nome, só para logs/CSV), `scheduler`/`seed` (a combinação corrente do loop externo — aqui sempre a seed vencedora escolhida na seção 1b, usada no bootstrap) e `decompose_fn` (a função de decomposição em si: `replace_layers_with_tucker` ou `replace_layers_with_tt_tensorly`). Para cada threshold de energia ainda não concluído: decompõe, mede compressão, GFLOPs e latência (1 imagem, via `benchmark_latency`/`count_macs`), e **avalia o modelo direto no conjunto de teste, sem fine-tuning** (zero-shot, com bootstrap) — e grava a linha imediatamente.

### Função `print_pareto_table`
Imprime a fronteira de Pareto (uma tabela) para os resultados de um método específico numa combinação `scheduler`/`seed` específica, incluindo GFLOPs e latência — juntando thresholds retomados de uma execução anterior com os calculados agora.

### Função `summarize_across_schedulers`
Lê `decomposition_sweep_pareto_resnet18_ensemble_no_ft.csv` inteiro e agrega, por `(method, threshold)`, média ± desvio padrão de cada métrica entre os 3 schedulers já concluídos — chamada uma vez, no final de `main()`. Substitui o `summarize_across_seeds` da versão anterior deste notebook: como agora cada scheduler já roda com sua melhor seed (seção 1b), a pergunta de estabilidade que faz sentido é "esse ponto do Pareto se mantém entre os 3 schedulers, ou só um deles favorece esse threshold?", não mais "entre seeds do mesmo scheduler".

In [10]:
# --- Persistência incremental + resume (mesmo padrão dos pipeline_resnetXX*.ipynb) ---
# Cada linha do sweep (um threshold de um método, para um par scheduler/seed) é gravada
# no CSV assim que fica pronta, em vez de acumular tudo em memória e escrever só no final
# de main(). Ao reexecutar, combinações (method, threshold, seed, scheduler) já presentes
# no CSV são puladas.
#
# Nome de arquivo próprio (sufixo "_ensemble_no_ft") para não colidir com os CSVs dos
# outros pipelines de decomposição (com FT, ou com checkpoints do Optuna): as colunas
# mudaram (sem tempo_treino_s, com scheduler) e os números não são comparáveis entre si
# (aqui é avaliação zero-shot, sem re-treino, sobre checkpoints da ablation ensemble).
OUT_CSV = OUTPUT_DIR / "decomposition_sweep_pareto_resnet18_ensemble_no_ft.csv"
CSV_FIELDNAMES = ["timestamp", "scheduler", "seed", "method", "fold", "baseline_f1", "baseline_auc", "baseline_params_m",
                  "baseline_gflops", "baseline_latency_ms", "baseline_tempo_teste_s",
                  "threshold", "ratio", "recon_err", "params_m", "size_mb",
                  "gflops", "gflops_ratio", "latency_ms", "speedup",
                  "test_f1", "test_auc", "auc_lo", "auc_hi", "delta_auc", "d_lo", "d_hi", "sig", "status",
                  "tempo_decomposicao_s", "tempo_teste_s"]
_ROW_FLOAT_COLS = ["threshold", "ratio", "recon_err", "params_m", "size_mb", "gflops", "gflops_ratio",
                   "latency_ms", "speedup", "test_f1", "test_auc", "auc_lo", "auc_hi",
                   "delta_auc", "d_lo", "d_hi", "tempo_decomposicao_s", "tempo_teste_s"]


def log_row(row: dict, csv_path=OUT_CSV):
    """Grava UMA linha (um threshold de um método, de um par scheduler/seed) assim que ela fica pronta."""
    exists = os.path.isfile(csv_path)
    if exists:
        with open(csv_path, newline="", encoding="utf-8") as f:
            header = next(csv.reader(f), [])
        if header != CSV_FIELDNAMES:
            raise ValueError(
                f"CSV existente tem colunas incompatíveis com o pipeline atual: {csv_path}"
            )
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def get_completed_thresholds(csv_path=OUT_CSV) -> set:
    """Lê o CSV (se existir) e retorna o conjunto de (method, threshold, seed, scheduler) já
    concluídos, para que run_sweep() pule essas combinações ao retomar um
    sweep interrompido no meio (crash, falta de energia, etc.)."""
    if not os.path.isfile(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {(row["method"], float(row["threshold"]), int(row["seed"]), row["scheduler"]) for row in reader}


def load_completed_rows(method, seed, scheduler, csv_path=OUT_CSV):
    """Recarrega do CSV as linhas já concluídas de um método NESSE par scheduler/seed, no
    mesmo formato de dict que run_sweep() produz, para que print_pareto_table() e o
    resumo final mostrem a fronteira de Pareto completa mesmo quando parte dela
    veio de uma execução anterior (thresholds pulados por já estarem prontos)."""
    if not os.path.isfile(csv_path):
        return []
    out = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for raw in reader:
            if raw["method"] != method or int(raw["seed"]) != seed or raw["scheduler"] != scheduler:
                continue
            r = {"method": method, "status": raw["status"], "sig": raw["sig"] == "True"}
            for c in _ROW_FLOAT_COLS:
                v = raw.get(c, "")
                r[c] = float(v) if v not in ("", None) else None
            if r["threshold"] not in THRESHOLDS:
                continue  # linha de um threshold que não existe mais na config atual
            out.append(r)
    return out

In [11]:
def run_sweep(method, scheduler, seed, decompose_fn, original, params_orig, size_orig, gflops_orig, lat_orig,
              base, base_lo, base_hi, test_loader, num_classes, classes, completed, base_row):
    print(f"\n{'='*60}\n MÉTODO = {method.upper()} | SCHEDULER = {scheduler} | SEED = {seed} | SEM FINE-TUNING\n{'='*60}")

    # Resume: recarrega do CSV o que já foi concluído para este método NESSE par
    # scheduler/seed antes de rodar mais nada — assim print_pareto_table() no final
    # mostra a fronteira inteira, mesmo que parte dela venha de uma execução anterior.
    rows = load_completed_rows(method, seed, scheduler)
    if rows:
        done = sorted(r["threshold"] for r in rows)
        print(f"[resume] {len(rows)} threshold(s) de '{method}' (scheduler={scheduler}, seed={seed}) já "
              f"concluído(s) em {OUT_CSV}; serão pulados: {done}")

    for thr in THRESHOLDS:
        if (method, thr, seed, scheduler) in completed:
            print(f"[SKIP] [{method}] scheduler={scheduler} seed={seed} threshold={thr} já está em {OUT_CSV}")
            continue

        print(f"\n{'#'*60}\n# [{method}] scheduler={scheduler} seed={seed} THRESHOLD DE ENERGIA = {thr}\n{'#'*60}")
        model = copy.deepcopy(original)            # cópia na CPU
        _, decomp_time = timed_call(decompose_fn, model, thr)
        model.to(device)

        params_comp = count_parameters(model)
        size_comp = get_model_size_mb(model)
        ratio = size_orig / size_comp
        errs = per_layer_reconstruction_error(original, model, method)
        err_med = float(np.mean(list(errs.values()))) if errs else float('nan')

        # Estrutura já está fixa aqui (não há mais fine-tuning que possa mudar
        # shapes/FLOPs) -> mede GFLOPs e latência (1 imagem, com warmup).
        gflops_comp = macs_to_gflops(count_macs(model, device, LATENCY_SHAPE))
        gflops_ratio = gflops_orig / gflops_comp if gflops_comp > 0 else float('nan')
        lat_comp = benchmark_latency(model, device, LATENCY_SHAPE, use_amp=USE_AMP)
        speedup = lat_orig['median_ms'] / lat_comp['median_ms']

        print(f"[compressão] params {params_orig/1e6:.2f}M -> {params_comp/1e6:.2f}M | "
              f"RAM {size_orig:.2f}->{size_comp:.2f} MB (ratio {ratio:.2f}x) | erro recon. médio {err_med:.1f}%")
        print(f"[custo] GFLOPs {gflops_orig:.2f} -> {gflops_comp:.2f} ({gflops_ratio:.2f}x) | "
              f"latência 1 img {lat_orig['median_ms']:.2f} -> {lat_comp['median_ms']:.2f} ms ({speedup:.2f}x)")
        print(f"Tempo de decomposição (wall-clock): {decomp_time:.2f}s")
        status = "ok" if ratio >= 1 else "inchou"
        if status == "inchou":
            print(f"[aviso] threshold {thr} INCHOU o modelo (ratio {ratio:.2f}x) — sem fine-tuning para "
                  f"recuperar acurácia, o teste abaixo tende a refletir isso.")

        # SEM fine-tuning: avalia direto no teste com os pesos reconstruídos pela
        # decomposição (zero-shot).
        res, test_time = timed_call(evaluate, model, test_loader, num_classes)
        auc_lo, auc_hi = bootstrap_auc_ci(res['y'], res['probs'], num_classes)
        d_lo, d_hi, sig = paired_delta_auc_ci(base['y'], base['probs'], res['probs'], num_classes)
        delta = res['auc'] - base['auc']
        sig_txt = "SIGNIFICATIVO" if sig else "n.s."
        print(f"[teste@{thr}] F1={res['f1']:.4f} | AUC={res['auc']:.4f} [IC95% {auc_lo:.4f}–{auc_hi:.4f}] "
              f"| ΔAUC={delta:+.4f} [IC95% {d_lo:+.4f}–{d_hi:+.4f}] {sig_txt}")
        print(f"   Classes: {classes}")
        print(f"   CM:\n{res['cm']}")
        print(f"   Tempo (wall-clock, informativo — NÃO é latência): decomposição {decomp_time:.2f}s | "
              f"teste (test set completo) {test_time:.2f}s")

        row = dict(method=method, threshold=thr, ratio=round(ratio, 3), recon_err=round(err_med, 2),
                   params_m=round(params_comp/1e6, 2), size_mb=round(size_comp, 2),
                   gflops=round(gflops_comp, 3), gflops_ratio=round(gflops_ratio, 3),
                   latency_ms=round(lat_comp['median_ms'], 3), speedup=round(speedup, 3),
                   test_f1=round(res['f1'], 4), test_auc=round(res['auc'], 4),
                   auc_lo=round(auc_lo, 4), auc_hi=round(auc_hi, 4),
                   delta_auc=round(delta, 4), d_lo=round(d_lo, 4), d_hi=round(d_hi, 4),
                   sig=sig, status=status,
                   tempo_decomposicao_s=round(decomp_time, 3), tempo_teste_s=round(test_time, 3))
        rows.append(row)
        log_row({**base_row, **row})
        free_gpu(model)

    rows.sort(key=lambda r: THRESHOLDS.index(r["threshold"]))
    return rows


def print_pareto_table(method, scheduler, seed, rows, base, base_lo, base_hi, params_orig, gflops_orig, lat_orig):
    print("\n" + "=" * 132)
    print(f" FRONTEIRA DE PARETO [{method}] scheduler={scheduler} seed={seed} SEM FT — baseline AUC={base['auc']:.4f} "
          f"[IC {base_lo:.4f}–{base_hi:.4f}], F1={base['f1']:.4f}, {params_orig/1e6:.2f}M params, "
          f"{gflops_orig:.2f} GFLOPs, {lat_orig['median_ms']:.2f} ms (1 img)")
    print("=" * 132)
    hdr = (f"{'thr':>6} | {'ratio':>5} | {'GFLOPs':>7} | {'gf_x':>5} | {'lat_ms':>7} | {'speedup':>7} | "
           f"{'recon%':>6} | {'Par_M':>5} | {'MB':>6} | "
           f"{'F1':>6} | {'AUC':>6} | {'AUC IC95%':>15} | {'ΔAUC':>7} | {'sig?':>4} | status")
    print(hdr); print("-" * len(hdr))
    for r in rows:
        gf  = f"{r['gflops']:.2f}"        if r['gflops']       is not None else "-"
        gfx = f"{r['gflops_ratio']:.2f}"  if r['gflops_ratio'] is not None else "-"
        lat = f"{r['latency_ms']:.2f}"    if r['latency_ms']   is not None else "-"
        spd = f"{r['speedup']:.2f}x"      if r['speedup']      is not None else "-"
        ic  = f"{r['auc_lo']:.3f}-{r['auc_hi']:.3f}" if r['auc_lo'] is not None else "-"
        f1_txt    = f"{r['test_f1']:.4f}"    if r['test_f1']    is not None else "-"
        auc_txt   = f"{r['test_auc']:.4f}"   if r['test_auc']   is not None else "-"
        delta_txt = f"{r['delta_auc']:+.4f}" if r['delta_auc']  is not None else "-"
        sig_txt   = ('sim' if r['sig'] else 'não') if r['sig'] is not None else "-"
        print(f"{r['threshold']:>6} | {r['ratio']:>5} | {gf:>7} | {gfx:>5} | {lat:>7} | {spd:>7} | "
              f"{r['recon_err']:>6} | {r['params_m']:>5} | {r['size_mb']:>6} | "
              f"{f1_txt:>6} | {auc_txt:>6} | {ic:>15} | {delta_txt:>7} | {sig_txt:>4} | {r['status']}")

In [12]:
def summarize_across_schedulers(csv_path=OUT_CSV, out_csv=None):
    """Agrega, por (method, threshold), média ± desvio padrão de cada métrica do
    sweep entre os schedulers concluídos em `csv_path`. Mede o quão estável é
    cada ponto do Pareto entre os 3 schedulers (cada um já com sua melhor seed,
    seção 1b) — ratio/AUC com desvio alto é menos confiável do que um com desvio
    baixo, mesmo com a mesma média. Chamada uma vez, no final de `main()`,
    depois que todos os schedulers em SCHEDULERS já rodaram."""
    if out_csv is None:
        out_csv = OUTPUT_DIR / "decomposition_sweep_summary_across_schedulers.csv"
    if not os.path.isfile(csv_path):
        print(f"[summarize_across_schedulers] {csv_path} não existe ainda — nada para agregar.")
        return []

    with open(csv_path, newline="", encoding="utf-8") as f:
        raw_rows = list(csv.DictReader(f))
    if not raw_rows:
        print(f"[summarize_across_schedulers] {csv_path} está vazio — nada para agregar.")
        return []

    metric_cols = ["ratio", "recon_err", "params_m", "size_mb", "gflops", "gflops_ratio",
                   "latency_ms", "speedup", "test_f1", "test_auc", "delta_auc"]

    groups = {}
    for raw in raw_rows:
        key = (raw["method"], float(raw["threshold"]))
        groups.setdefault(key, []).append(raw)

    method_order = {m: i for i, m in enumerate(METHODS)}
    summary = []
    for (method, thr), rs in sorted(groups.items(), key=lambda kv: (method_order.get(kv[0][0], 99), kv[0][1])):
        entry = dict(method=method, threshold=thr, n_schedulers=len(rs),
                     schedulers=sorted({r["scheduler"] for r in rs}))
        for c in metric_cols:
            vals = np.array([float(r[c]) for r in rs if r.get(c) not in ("", None)])
            entry[f"{c}_mean"] = float(vals.mean()) if len(vals) else float('nan')
            entry[f"{c}_std"]  = float(vals.std())  if len(vals) else float('nan')
        n_sig = sum(1 for r in rs if r.get("sig") == "True")
        entry["sig_frac"] = n_sig / len(rs) if rs else float('nan')
        summary.append(entry)

    print("\n" + "=" * 118)
    print(f" RESUMO ENTRE SCHEDULERS ({len(SCHEDULERS)} configurados) — média ± desvio padrão por (method, threshold)")
    print("=" * 118)
    hdr = (f"{'method':<13} | {'thr':>5} | {'n':>2} | {'ratio':>13} | {'gf_x':>13} | "
           f"{'speedup':>13} | {'F1':>13} | {'AUC':>13} | {'ΔAUC':>13} | {'sig%':>5}")
    print(hdr); print("-" * len(hdr))

    def fmt(entry, c, prec=3):
        m, s = entry[f"{c}_mean"], entry[f"{c}_std"]
        return "-" if np.isnan(m) else f"{m:.{prec}f}±{s:.{prec}f}"

    for e in summary:
        print(f"{e['method']:<13} | {e['threshold']:>5} | {e['n_schedulers']:>2} | "
              f"{fmt(e, 'ratio'):>13} | {fmt(e, 'gflops_ratio'):>13} | {fmt(e, 'speedup'):>13} | "
              f"{fmt(e, 'test_f1', 4):>13} | {fmt(e, 'test_auc', 4):>13} | {fmt(e, 'delta_auc', 4):>13} | "
              f"{e['sig_frac']*100:>4.0f}%")

    fieldnames = ["method", "threshold", "n_schedulers", "schedulers"] + \
                 [f"{c}_{stat}" for c in metric_cols for stat in ("mean", "std")] + ["sig_frac"]
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for e in summary:
            row = {k: e[k] for k in fieldnames if k != "schedulers"}
            row["schedulers"] = ";".join(str(s) for s in e["schedulers"])
            w.writerow(row)
    print(f"\n[MLOps] resumo entre schedulers salvo em: {out_csv}")
    return summary

## 9. Main — roda o sweep completo (Tucker + Tensor-Train) para cada scheduler, com resume

### Função `main`
Orquestra o experimento completo para **todos** os schedulers em `SCHEDULERS`, cada um com sua seed vencedora (seção 1b): carrega o `ImageFolder` uma única vez (as imagens/rótulos não mudam), mas dentro do loop refaz o split de teste com `random_state=seed` (a mesma seed usada para treinar aquele checkpoint, sem leakage), carrega o checkpoint `{scheduler}_seed{seed}_fold{TARGET_FOLD}_best.pth` correspondente e mede um baseline próprio (F1/AUC/GFLOPs/latência) para essa combinação. Para cada uma, roda `run_sweep` para cada método em `METHODS` (Tucker e Tensor-Train), reaproveitando o mesmo baseline/split/loader — a única coisa que varia entre os dois métodos é a função de decomposição. Como não há fine-tuning, não existe split de treino/validação: só o holdout de teste é necessário. Cada linha do sweep já é gravada incrementalmente dentro de `run_sweep` (seção 8), com resume automático por `(method, threshold, seed, scheduler)`, então `main()` só reporta o total ao final. Depois que os 3 schedulers rodaram, chama `summarize_across_schedulers()` (média ± desvio padrão entre os 3) e, por último, `compare_single_image_latency` para a comparação de inferência de 1 imagem entre original, Tucker e TT.

In [13]:
def main():
    print(f"--- Sweep Tucker + Tensor-Train (ResNet-18 ensemble, SEM fine-tuning) | {device} | "
          f"AMP={USE_AMP} | cudnn.benchmark={not DETERMINISTIC} ---")
    print(f"[schedulers] {SCHEDULERS} | [seed vencedora por scheduler] {BEST_SEED_PER_SCHEDULER}")

    # "Originais" é uma pasta plana (healthy/mild/moderate/severe), sem divisão prévia
    # em train/test. O ImageFolder em si (imagens/rótulos) não depende da seed — só o
    # split train/test (dentro do loop, abaixo) usa a seed de cada checkpoint. Sem
    # fine-tuning não há mais treino, então só carregamos a versão sem augmentation
    # (usada tanto para o baseline quanto para o sweep).
    full_data_plain = datasets.ImageFolder(BASE_DIR, data_transforms['teste'])
    num_classes = len(full_data_plain.classes)
    print(f"[class_to_idx] {full_data_plain.class_to_idx}")
    all_labels = np.array(full_data_plain.targets)

    # Resume: pula (method, threshold, seed, scheduler) que já estão em OUT_CSV, para que
    # um sweep interrompido no meio (crash, falta de energia, etc.) não precise
    # redecompor nem reavaliar do zero o que já foi concluído.
    completed = get_completed_thresholds()
    if completed:
        print(f"[resume] {len(completed)} combinação(ões) (método, threshold, seed, scheduler) já "
              f"concluída(s) em {OUT_CSV}; serão puladas.")

    all_rows = []
    last_original, last_num_classes = None, num_classes

    for scheduler in SCHEDULERS:
        seed = BEST_SEED_PER_SCHEDULER[scheduler]
        ckpt = CHECKPOINT_DIR / f"{scheduler}_seed{seed}_fold{TARGET_FOLD}_best.pth"
        if not os.path.exists(ckpt):
            raise FileNotFoundError(f"{ckpt} não encontrado. Rode pipeline_resnet18.ipynb primeiro "
                                    f"(scheduler {scheduler}, fold {TARGET_FOLD}, seed {seed}).")

        print(f"\n{'*'*70}\n SCHEDULER = {scheduler}  |  SEED = {seed}  (checkpoint: {ckpt.name})\n{'*'*70}")
        set_seed(seed)

        # Holdout estratificado 80/20: MESMA seed usada ao treinar esse checkpoint, então
        # este test_idx reproduz o conjunto blindado original daquela seed (sem leakage).
        # Os 80% de cross-validation não são usados aqui (sem treino/FT).
        _, test_idx = train_test_split(
            np.arange(len(all_labels)), test_size=TEST_SIZE,
            random_state=seed, stratify=all_labels)
        print(f"[split] teste: {len(test_idx)} imgs (20%) — os outros 80% não são usados (sem fine-tuning)")

        test_data = Subset(full_data_plain, test_idx)
        persistent = NUM_WORKERS > 0
        test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=persistent)

        # Original carregado na CPU; sobe à GPU só para o baseline, depois volta p/ CPU.
        original = build_model(num_classes)
        original.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=True))
        original.eval()
        params_orig = count_parameters(original)
        size_orig = get_model_size_mb(original)
        orig_tmp = OUTPUT_DIR / f"_orig_tmp_resnet18_ensemble_no_ft_{scheduler}_seed{seed}.pth"
        torch.save(original.state_dict(), orig_tmp)
        disk_orig = os.path.getsize(orig_tmp) / 1024**2
        os.remove(orig_tmp)

        original.to(device)
        base, base_test_time = timed_call(evaluate, original, test_loader, num_classes)
        base_lo, base_hi = bootstrap_auc_ci(base['y'], base['probs'], num_classes)

        # GFLOPs e latência de referência (1 imagem, com warmup) — medidos aqui, com o
        # modelo ainda na GPU, ANTES do sweep, para servir de baseline de comparação.
        gflops_orig = macs_to_gflops(count_macs(original, device, LATENCY_SHAPE))
        lat_orig = benchmark_latency(original, device, LATENCY_SHAPE, use_amp=USE_AMP)

        original.to('cpu')                     # libera a VRAM durante o sweep
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("\n" + "=" * 60)
        print(f"[BASELINE scheduler={scheduler} seed={seed}] fold-{TARGET_FOLD}: F1={base['f1']:.4f} | "
              f"AUC={base['auc']:.4f} [IC95% {base_lo:.4f}–{base_hi:.4f}]")
        print(f"Recursos: {params_orig/1e6:.2f}M params | {size_orig:.2f} MB (RAM) | {disk_orig:.2f} MB (disco) | "
              f"{gflops_orig:.2f} GFLOPs | {lat_orig['median_ms']:.2f} ms (1 img, mediana de 50 runs)")
        print(f"Tempo (wall-clock, informativo — NÃO é latência): teste (test set completo) {base_test_time:.2f}s")
        print("=" * 60)

        # Campos fixos de cada linha do CSV (repetidos em toda linha do sweep DESSE
        # scheduler, já que o baseline é medido uma vez por combinação e reaproveitado
        # pelos dois métodos).
        base_row = dict(timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        scheduler=scheduler, seed=seed, fold=TARGET_FOLD, baseline_f1=round(base['f1'], 4),
                        baseline_auc=round(base['auc'], 4), baseline_params_m=round(params_orig/1e6, 2),
                        baseline_gflops=round(gflops_orig, 3),
                        baseline_latency_ms=round(lat_orig['median_ms'], 3),
                        baseline_tempo_teste_s=round(base_test_time, 3))

        # === roda o sweep de cada método de decomposição, reaproveitando baseline/dados/loader desse scheduler ===
        for method in METHODS:
            rows = run_sweep(method, scheduler, seed, DECOMPOSERS[method], original, params_orig, size_orig,
                             gflops_orig, lat_orig,
                             base, base_lo, base_hi, test_loader,
                             num_classes, full_data_plain.classes,
                             completed, base_row)
            print_pareto_table(method, scheduler, seed, rows, base, base_lo, base_hi, params_orig, gflops_orig, lat_orig)
            all_rows.extend(rows)

        last_original, last_num_classes = original, num_classes

    # Cada linha já foi gravada incrementalmente em OUT_CSV por log_row() dentro de
    # run_sweep() (tanto as novas quanto as retomadas de uma execução anterior) —
    # nada para escrever aqui, só reportar o total.
    total_expected = len(SCHEDULERS) * len(METHODS) * len(THRESHOLDS)
    print(f"\n[MLOps] Pareto combinado (Tucker + Tensor-Train, sem fine-tuning) em: {OUT_CSV} "
          f"({len(all_rows)}/{total_expected} combinações método×scheduler×threshold — gravado "
          f"incrementalmente, linha a linha, com resume automático).")
    print("\nLeitura: para cada método, escolha o maior 'ratio' cujo ΔAUC seja 'não' significativo. "
          "Como não há fine-tuning, ΔAUC aqui mede o impacto PURO da decomposição sobre a acurácia "
          "(sem recuperação via treino) — compare Tucker vs Tensor-Train no MESMO threshold: Tucker "
          "mantém 3 convoluções reais (sem reconstruir o kernel denso) e por isso REDUZ GFLOPs de "
          "verdade; TT reconstrói o kernel denso a partir dos núcleos — comprime parâmetros/RAM, mas "
          "os GFLOPs praticamente não caem (gf_x perto de 1.00x). Use a coluna 'lat_ms'/'speedup' — "
          "não os tempos de wall-clock do test set — para comparar velocidade de inferência real. Para "
          "ver o quão estável cada ponto é entre os 3 schedulers, veja o resumo de "
          "summarize_across_schedulers() logo abaixo.")

    # === Agrega média ± desvio padrão de cada métrica por (method, threshold) entre os schedulers concluídos ===
    summarize_across_schedulers()

    # === Comparação final: inferência de 1 imagem, original vs Tucker vs Tensor-Train ===
    # Latência/GFLOPs dependem só da arquitetura (shapes), não dos valores dos pesos —
    # então qualquer combinação com checkpoint disponível serve; usamos a última carregada.
    if last_original is not None:
        compare_single_image_latency(last_original, last_num_classes)
    else:
        print("\n[aviso] nenhum scheduler teve checkpoint disponível — pulando comparação de latência de 1 imagem.")

## 10. Comparação final — inferência de 1 imagem (original vs Tucker vs Tensor-Train)

### Função `compare_single_image_latency`
Chamada ao final de `main()`. Reconstrói a arquitetura Tucker e a arquitetura TT no threshold `SINGLE_IMAGE_THRESHOLD` a partir do `original` (sem carregar nenhum checkpoint fine-tunado — não existe mais, já que o sweep não faz FT). A latência e os GFLOPs não dependem dos valores dos pesos, só da arquitetura/shapes, então os pesos reconstruídos pela decomposição já são suficientes. Mede a latência de **uma única imagem** (`benchmark_latency`, batch=1, com warmup) e os GFLOPs de cada um dos três modelos, imprime uma tabela comparativa e salva em `single_image_latency_comparison_resnet18_ensemble_no_ft.csv`.

In [14]:
def compare_single_image_latency(original, num_classes, threshold=SINGLE_IMAGE_THRESHOLD,
                                 warmup=10, runs=50):
    """Compara, para 1 ÚNICA imagem (batch=1), a latência de inferência da rede
    original, da versão Tucker e da versão Tensor-Train (ambas decompostas no
    threshold `SINGLE_IMAGE_THRESHOLD`, SEM fine-tuning), além dos GFLOPs de cada uma."""
    print("\n" + "=" * 70)
    print(f" INFERÊNCIA DE 1 IMAGEM — original vs Tucker vs Tensor-Train (thr={threshold}, sem FT)")
    print("=" * 70)

    rows = []

    original.to(device).eval()
    lat = benchmark_latency(original, device, LATENCY_SHAPE, warmup=warmup, runs=runs, use_amp=USE_AMP)
    gflops = macs_to_gflops(count_macs(original, device, LATENCY_SHAPE))
    rows.append(dict(model="original", latency_ms=lat['median_ms'], gflops=gflops))
    original.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for method in METHODS:
        model = copy.deepcopy(original)
        DECOMPOSERS[method](model, threshold)
        model.to(device).eval()

        lat = benchmark_latency(model, device, LATENCY_SHAPE, warmup=warmup, runs=runs, use_amp=USE_AMP)
        gflops = macs_to_gflops(count_macs(model, device, LATENCY_SHAPE))
        rows.append(dict(model=method, latency_ms=lat['median_ms'], gflops=gflops))
        free_gpu(model)

    base_lat = rows[0]['latency_ms']
    hdr = f"{'modelo':<14} | {'lat_ms (1 img)':>14} | {'GFLOPs':>8} | {'speedup':>8}"
    print(hdr); print("-" * len(hdr))
    for r in rows:
        speedup = base_lat / r['latency_ms']
        print(f"{r['model']:<14} | {r['latency_ms']:>14.3f} | {r['gflops']:>8.3f} | {speedup:>7.2f}x")

    out_csv = OUTPUT_DIR / "single_image_latency_comparison_resnet18_ensemble_no_ft.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["timestamp", "model", "threshold", "latency_ms", "gflops",
                                          "speedup_vs_original"])
        w.writeheader()
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        for r in rows:
            w.writerow(dict(timestamp=ts, model=r['model'],
                            threshold=(threshold if r['model'] != 'original' else None),
                            latency_ms=round(r['latency_ms'], 4), gflops=round(r['gflops'], 4),
                            speedup_vs_original=round(base_lat / r['latency_ms'], 3)))
    print(f"\n[MLOps] comparação de latência (1 imagem) salva em: {out_csv}")
    return rows

## Execução

Roda `main()` uma vez, cobrindo os 3 schedulers em `SCHEDULERS` (cada um com sua seed vencedora da seção 1b) sequencialmente por dentro. Cada scheduler roda `METHODS` × `THRESHOLDS` = 24 pontos, então até 72 linhas no total em `decomposition_sweep_pareto_resnet18_ensemble_no_ft.csv`. Se o checkpoint da seed vencedora de algum scheduler não existir em `CHECKPOINT_DIR`, `main()` levanta `FileNotFoundError` e interrompe nesse ponto — os schedulers anteriores já ficam gravados no CSV; re-executar a célula retoma dali via resume.

In [15]:
if __name__ == '__main__':
    torch.multiprocessing.freeze_support()
    main()

--- Sweep Tucker + Tensor-Train (ResNet-18 ensemble, SEM fine-tuning) | cuda | AMP=True | cudnn.benchmark=True ---
[schedulers] ['CosineAnnealingLR', 'OneCycleLR', 'ReduceLROnPlateau'] | [seed vencedora por scheduler] {'CosineAnnealingLR': 123, 'OneCycleLR': 123, 'ReduceLROnPlateau': 555}
[class_to_idx] {'healthy': 0, 'mild': 1, 'moderate': 2, 'severe': 3}

**********************************************************************
 SCHEDULER = CosineAnnealingLR  |  SEED = 123  (checkpoint: CosineAnnealingLR_seed123_fold1_best.pth)
**********************************************************************
[split] teste: 92 imgs (20%) — os outros 80% não são usados (sem fine-tuning)

[BASELINE scheduler=CosineAnnealingLR seed=123] fold-1: F1=0.7467 | AUC=0.9211 [IC95% 0.8782–0.9586]
Recursos: 11.18M params | 42.68 MB (RAM) | 42.72 MB (disco) | 3.63 GFLOPs | 3.55 ms (1 img, mediana de 50 runs)
Tempo (wall-clock, informativo — NÃO é latência): teste (test set completo) 7.05s

 MÉTODO = TUCKER | SC